In [98]:
import pandas as pd
import numpy as np 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor as gpr
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


In [99]:
SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

In [100]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 6):
    n_data = Dataset[Dataset["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

### Treino:
- X → normalização → PCA (fit) → X_pca → GPR (fit)

### Teste:
- X → normalização (transform) → PCA (transform) → X_pca → GPR (predict)

In [101]:
def CreatePCAdf(pca, predictors):
    # Matriz de transformação do PCA
    W = pca.components_.T   # shape (n_variaveis, n_componentes)

    # Nomes das componentes
    cp_names = [f"CP{i+1}" for i in range(W.shape[1])]

    # Criar DataFrame
    df_pca = pd.DataFrame(
        data=np.round(W, 3),
        index=predictors,
        columns=cp_names
    )

    return df_pca    

In [102]:
def TransformPCA(X_train, X_test, n_comp, predictors):
    pca = PCA(n_components=n_comp)
    
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca  = pca.transform(X_test)
    df = CreatePCAdf(pca, predictors)

    total = np.round(np.sum(pca.explained_variance_ratio_) * 100, 3)

    if total < 90:
        df, pca, X_train_pca, X_test_pca, n_comp = TransformPCA(X_train, X_test, n_comp + 1, predictors)

    print(f"Variância (%): {np.round(pca.explained_variance_ratio_ * 100, 3)}")    
    print(f"Total (%): {total}")
    
    return df, pca, X_train_pca, X_test_pca, n_comp

In [103]:
import matplotlib.pyplot as plt
import numpy as np
import os

def PlotPredictions(train_orig, train_pred, test_orig, test_pred, target_name, n): 
    # cria figura
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -----------------------------
    # SUBPLOT 1 — TREINO
    # -----------------------------
    ax = axes[0]
    n_train = len(train_orig)
    x_train = np.arange(n_train)

    ax.plot(x_train, train_orig, label="Original (train)", color="blue", )
    ax.plot(x_train, train_pred, label="Predito (train)", color="red",)

    ax.scatter(x_train, train_orig, color="blue", s=35)
    ax.scatter(x_train, train_pred, color="red", s=35)

    ax.set_title("Treinamento")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    # -----------------------------
    # SUBPLOT 2 — TESTE
    # -----------------------------
    ax = axes[1]
    n_test = len(test_orig)
    x_test = np.arange(n_test)

    ax.plot(x_test, test_orig, label="Original (test)", color="blue", linewidth=1.8)
    ax.plot(x_test, test_pred, label="Predito (test)", color="red", linewidth=1.8)

    ax.scatter(x_test, test_orig, color="blue", s=35)
    ax.scatter(x_test, test_pred, color="red", s=35)

    ax.set_title("Teste")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    plt.tight_layout()

    # salva a figura
    filename = f"./Dados/VirtualData/P{n}/TrainResults/{target_name}.pdf"
    plt.savefig(filename, format="pdf", bbox_inches="tight")

    # fecha a figura (IMPORTANTE para não acumular memória)
    plt.close(fig)

    print(f"Figura salva em: {filename}")


In [104]:
def PlotVirtualData(virtual_df, original_df, predictors, target, n):
    savepath = f"./Dados/VirtualData/P{n}/VSGResults/Virtual_{target}.pdf"

    total_features = len(predictors) + 1
    fig, axes = plt.subplots(total_features, 1, figsize=(10, 2*total_features), sharex=False)

    # junta entradas + saída
    features = predictors + [target]

    for i, feat in enumerate(features):

        ax = axes[i]

        # valores preditos (virtuais)
        y_pred = virtual_df[feat].values
        x_pred = np.arange(len(y_pred))

        # valores originais reais (Dataset)
        y_orig = original_df[feat].values
        x_orig = np.arange(len(y_orig))

        # plot
        ax.plot(x_pred, y_pred, color="red", label="Virtual (predito)", linewidth=1.7)
        ax.scatter(x_pred, y_pred, color="red", s=20)

        ax.plot(x_orig, y_orig, color="blue", label="Original", linewidth=1.7)
        ax.scatter(x_orig, y_orig, color="blue", s=20)

        ax.set_ylabel(feat)
        ax.grid(True)

        if i == 0:
            ax.legend()

    axes[-1].set_xlabel("Amostras")

    plt.tight_layout()
    plt.savefig(savepath, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Figura salva em: {savepath}")


In [105]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

def ComputeMetrics(y_train, y_train_pred, y_test, y_test_pred):
    return {
        "mse_train": mean_squared_error(y_train, y_train_pred),
        "r2_train":  r2_score(y_train, y_train_pred),
        "mse_test":  mean_squared_error(y_test, y_test_pred),
        "r2_test":   r2_score(y_test, y_test_pred)
    }

In [106]:
GPR_PARAMS = {
    "Fe": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-8},
    "Al": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "As": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Pb": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Zn": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Hg": {"nu": 0.5, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Co": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e10, "alpha": 1e-3},
    "V":  {"nu": 0.25, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Ba": {"nu": 0.5,  "ls_min": 1e-8, "ls_max": 1e8, "alpha": 1e-3},
    "Mn": {"nu": 1.5,  "ls_min": 1e-8, "ls_max": 1e8, "alpha": 1e-3},
}

In [107]:
def GprModel(X_train_pca, X_test_pca, y_train, y_test, target, n):

    params = GPR_PARAMS[target]

    kernel = C(1.0, (1e-3, 1e3)) + C(1.0) * Matern(
        length_scale=np.ones(X_train_pca.shape[1]),
        nu=params["nu"],
        length_scale_bounds=(params["ls_min"], params["ls_max"])
    )

    model = gpr(
        kernel=kernel,
        alpha=params["alpha"],
        normalize_y=False,
        n_restarts_optimizer=20
    )

   # Treinamento
    model.fit(X_train_pca, y_train)

    # Predições
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)

    # Desnormalização
    y_train_denorm = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_train_pred_denorm = OUT_SCALER.inverse_transform(y_train_pred.reshape(-1, 1)).ravel()

    y_test_denorm = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_test_pred_denorm = OUT_SCALER.inverse_transform(y_test_pred.reshape(-1, 1)).ravel()
    
    metrics = ComputeMetrics(y_train_denorm, y_train_pred_denorm,  y_test_denorm, y_test_pred_denorm)
    PlotPredictions(y_train_denorm, y_train_pred_denorm,  y_test_denorm, y_test_pred_denorm, target, n)

    
    return  model, metrics

In [108]:
class GPRVSG:
    def __init__(self, model, x_train, y_train, target):
        self.model = model
        self.x_train = x_train
        self.y_train = y_train
        self.target = target

        self.d = self.x_train.shape[1]
        self.virtual_samples_x = []
        self.virtual_samples_y = None
        self.virtual_samples_df = None


    def ComputeProjection(self):
        projections = []
        for m in range(self.d):
            x_m = self.x_train[:, m]
            projections.append(np.sort(x_m))
        return projections


    def SetInputSpace(self):
        projections = self.ComputeProjection()
        avg_dists = [np.mean(np.diff(proj)) for proj in projections]

        Q_alpha = np.quantile(
            projections, [0.5], axis=1, method="hazen").T

        for m in range(self.d):
            for i in range(len(projections[m]) - 1):
                dist = projections[m][i + 1] - projections[m][i]

                if dist > avg_dists[m]:
                    G = 0.5 * (projections[m][i] + projections[m][i + 1])

                    for q in range(self.d):
                        if q != m:
                            for quantile in Q_alpha[q]:
                                tilde_q = np.zeros(self.d)
                                tilde_q[m] = G
                                tilde_q[q] = quantile
                                self.virtual_samples_x.append(tilde_q)

        self.virtual_samples_x = np.asarray(self.virtual_samples_x)

    def getX(self):
        n_dims = self.virtual_samples_x.shape[1]
        self.col_names = [f"x{i+1}" for i in range(n_dims)]
        df = pd.DataFrame(self.virtual_samples_x, columns=self.col_names)
        return [df[col] for col in self.col_names]


    def ComputeY(self,):
        # pega lista de colunas: [x1, x2, ..., xn]
        X_cols = self.getX()

        # monta matriz X de entrada
        X = np.column_stack(X_cols)

        data = {name: X_cols[i] for i, name in enumerate(self.col_names)}
        # prediz y
        y_pred = self.model.predict(X)
        y_pred = y_pred.reshape(-1, 1)
        data[self.target] = OUT_SCALER.inverse_transform(y_pred).ravel()

        self.virtual_samples_y = data[self.target]
        self.virtual_samples_df = pd.DataFrame(data)

    def Run(self):
            self.SetInputSpace()
            self.ComputeY()


In [109]:
def PCAInverse(pca, x, y, target, predictors):
    # volta do PCA para o espaço normalizado original
    X_reconstructed_scaled = pca.inverse_transform(x)

    # desfaz a normalização original dos preditores
    X_reconstructed = SCALER.inverse_transform(X_reconstructed_scaled)

    # monta DataFrame com nomes reais dos preditores
    df_vs = pd.DataFrame(X_reconstructed, columns=predictors)
    df_vs[target] = y
    return df_vs


In [111]:
ConfDataset = pd.read_excel("./GA-GPR.xlsx")
if "n_comp" not in ConfDataset.columns:
    ConfDataset["n_comp"] = np.nan

Results = {}

for i in range(np.size(ConfDataset["P"])):
    P = ConfDataset.iloc[i]["P"]
    target = ConfDataset.iloc[i]["target"]
    n_predictors = ConfDataset.iloc[i]["n_predictors"]
    predictors = (ConfDataset.iloc[i]["predictors"]).split("; ")
    Dataset = Datasets[P-1]
    
    if n_predictors > 3:
        n_predictors = 3
        
    print(f"++++++++++++++++++++++ Ponto {P}, target {target} ++++++++++++++++++++++++++")
    output_dir = f"./Dados/VirtualData/P{P}"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"./Dados/VirtualData/P{P}/TrainResults/", exist_ok=True)
    os.makedirs(f"./Dados/VirtualData/P{P}/VSGResults/", exist_ok=True)
    
    X = Dataset[predictors].values
    Y = Dataset[target].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

    X_train_scaled = SCALER.fit_transform(X_train)
    X_test_scaled  = SCALER.transform(X_test)
    
    df, pca, x_train, x_test, n_comp = TransformPCA(X_train_scaled, X_test_scaled, n_predictors, predictors)
    
    ConfDataset.loc[i, "n_comp"] = int(n_comp)
    ConfDataset.to_excel("./GA-GPR.xlsx", index=False)
    print(f" → {target}")      
    
    y_train = OUT_SCALER.fit_transform(Y_train.reshape(-1, 1)).ravel()
    y_test  = OUT_SCALER.transform(Y_test.reshape(-1, 1)).ravel()
    
    model, metrics = GprModel(x_train, x_test, y_train, y_test, target, P)
    gpr_vsg = GPRVSG(model, x_train, y_train, target)
    gpr_vsg.Run()
    
    vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
    df_vs = PCAInverse(pca, gpr_vsg.virtual_samples_x, gpr_vsg.virtual_samples_y, target, predictors)
    with pd.ExcelWriter(vs_filename, engine="openpyxl") as writer:
        df_vs.to_excel(writer, index=False, sheet_name="orig-vs")
        gpr_vsg.virtual_samples_df.to_excel(writer, index=False, sheet_name="pca-vs")
        print(f"{vs_filename} Exported !!!!")
        
    

    PlotVirtualData(
        virtual_df = df_vs,          
        original_df = Dataset,                       
        predictors = predictors,                          
        target = target,                                  
        n = P
    )
    
    Results[target] = metrics    
display(pd.DataFrame(Results).T)

++++++++++++++++++++++ Ponto 1, target Fe ++++++++++++++++++++++++++
Variância (%): [67.487 26.235  6.278]
Total (%): 100.0
 → Fe


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P1/TrainResults/Fe.pdf
./Dados/VirtualData/P1\virtual_samples_Fe.xlsx Exported !!!!
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Fe.pdf
++++++++++++++++++++++ Ponto 1, target Al ++++++++++++++++++++++++++
Variância (%): [42.361 20.796 15.362 11.026  8.147]
Total (%): 97.692
Variância (%): [42.361 20.796 15.362 11.026  8.147]
Total (%): 89.545
Variância (%): [42.361 20.796 15.362 11.026  8.147]
Total (%): 78.519
 → Al


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P1/TrainResults/Al.pdf
./Dados/VirtualData/P1\virtual_samples_Al.xlsx Exported !!!!
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Al.pdf
++++++++++++++++++++++ Ponto 1, target As ++++++++++++++++++++++++++
Variância (%): [54.486 45.514]
Total (%): 100.0
 → As


...

d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P5/TrainResults/Ba.pdf
./Dados/VirtualData/P5\virtual_samples_Ba.xlsx Exported !!!!
Figura salva em: ./Dados/VirtualData/P5/VSGResults/Virtual_Ba.pdf
++++++++++++++++++++++ Ponto 5, target Mn ++++++++++++++++++++++++++
Variância (%): [43.596 37.725 18.679]
Total (%): 100.0
 → Mn


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P5/TrainResults/Mn.pdf
./Dados/VirtualData/P5\virtual_samples_Mn.xlsx Exported !!!!
Figura salva em: ./Dados/VirtualData/P5/VSGResults/Virtual_Mn.pdf


,mse_train,r2_train,mse_test,r2_test
Fe,1.792566e-11,1.000000,65839.205802,0.731557
Al,1.439480e+00,0.999995,1553.358285,-1.355966
As,2.705968e-08,0.999999,0.027608,-0.328130
Pb,2.132940e-07,0.999999,0.068196,0.230514
Zn,1.479504e-03,0.999997,481.307222,-0.367840
Hg,4.076131e-09,0.999999,0.002212,-3.059359
Co,2.877384e-08,0.999998,0.026841,-0.223516
V,9.883240e-08,0.999999,0.015356,0.660247
Ba,6.607954e-03,0.999980,156.984755,0.822637
Mn,2.221349e-03,0.999998,2045.149792,0.222009
